# Notebook 04 — Fourier Seasonal Representation

**Reference:** West & Harrison, Chapter 8 (§8.4–8.6)

**New engine function:** `make_fourier_seasonal(period, n_harmonics, V, W_season)`

## 1. Two ways to model seasonality

The Beginner tutorial used the **factor form** (sum-to-zero constraint, state dim = period − 1). Its main weakness: the state dimension grows linearly with the period. A monthly series needs 11 state components even if the seasonal pattern is smooth and could be captured with far fewer parameters.

The **Fourier form** represents the seasonal pattern as a sum of sinusoids:

$$
s_t = \sum_{j=1}^{J} \left[a_j \cos(\omega_j t) + b_j \sin(\omega_j t)\right], \quad \omega_j = \frac{2\pi j}{s}
$$

with $J \le s/2$ harmonics. Each harmonic contributes 2 state dimensions (the sine and cosine amplitudes), so the total state dimension is $2J$ (or $2J-1$ if $J = s/2$ and $s$ is even, because the Nyquist harmonic degenerates to 1D).

| Approach | State dim | Parameters | Best for |
|----------|-----------|------------|----------|
| Factor form | $s-1$ | 1 ($W_{\text{season}}$) | Low-period, irregular patterns |
| Fourier form ($J$ harmonics) | $2J$ (or $2J-1$) | $J$ ($W_j$ per harmonic, or 1 shared) | Smooth patterns, long periods |

## 2. State-space representation of a harmonic

A single harmonic at frequency $\omega_j$ has the rotation-block structure:

$$
G_j = \begin{bmatrix} \cos\omega_j & \sin\omega_j \\ -\sin\omega_j & \cos\omega_j \end{bmatrix}
, \quad
F_j = \begin{bmatrix} 1 & 0 \end{bmatrix}
$$

The first component of the state is the cosine amplitude $a_j^{(t)}$; $G_j$ rotates it forward one step. Observe:

- $G_j G_j^\top = I$ (rotation → orthogonal matrix)
- $F_j G_j^k \mathbf{1}$ traces out $\cos(\omega_j k)$ for state initialised at $(1, 0)$

Multiple harmonics are superposed via the `combine()` function (block-diagonal $G$ and $W$).

In [ ]:
import sys
from pathlib import Path

project_root = Path().resolve().parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import matplotlib.pyplot as plt

from engine.filter import kalman_filter
from engine.models import make_fourier_seasonal, make_seasonal_factor
from engine.simulate import simulate

## 3. State dimension as a function of J

In [ ]:
# Monthly series: period = 12, max harmonics = 6
period = 12
print(f"Period = {period}")
print(f"Factor form state dim: {period - 1}")
print()
for J in range(1, period // 2 + 1):
    spec = make_fourier_seasonal(period=period, n_harmonics=J, V=1.0, W_season=0.01)
    print(f"J={J} harmonics → state dim = {spec.d}")

## 4. Rotation block verification

In [ ]:
# J=1 Fourier seasonal: G should be a 2x2 rotation matrix
spec_j1 = make_fourier_seasonal(period=12, n_harmonics=1, V=1.0, W_season=0.01)
G = spec_j1.G

omega = 2 * np.pi / 12
G_expected = np.array([[np.cos(omega), np.sin(omega)],
                        [-np.sin(omega), np.cos(omega)]])

print("G (J=1, period=12):")
print(np.round(G, 4))
print("\nG expected:")
print(np.round(G_expected, 4))
print(f"\nG G' = I: {np.allclose(G @ G.T, np.eye(2))}")

## 5. Fitting a seasonal series — effect of J

In [ ]:
# Simulate a quarterly seasonal series (period=4) with 1 harmonic (smooth)
V_true, W_true = 0.5, 0.02
spec_true = make_seasonal_factor(period=4, V=V_true, W_season=W_true)
sim = simulate(spec_true, n=120, seed=5)
y = sim.y

# Fit Fourier seasonal models with J=1, 2 (max for period=4)
results = {}
for J in [1, 2]:
    spec_f = make_fourier_seasonal(period=4, n_harmonics=J, V=V_true, W_season=W_true)
    fr = kalman_filter(spec_f, y)
    results[J] = (spec_f, fr)
    print(f"J={J}: state dim={spec_f.d}, loglik={fr.loglik:.2f}")

# Also the factor-form baseline
fr_factor = kalman_filter(spec_true, y)
print(f"Factor form: state dim={spec_true.d}, loglik={fr_factor.loglik:.2f}")

In [ ]:
t = np.arange(len(y))

fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
labels = ['Fourier J=1', 'Fourier J=2', 'Factor form (baseline)']
filters = [results[1][1], results[2][1], fr_factor]

for ax, fr, label in zip(axes, filters, labels):
    ax.plot(t, y[:, 0], 'o', ms=2, alpha=0.3, color='grey', label='obs')
    ax.plot(t, fr.f[:, 0], 'C0', lw=1.2, label=f'one-step forecast ({label})')
    ax.set_ylabel('y')
    ax.legend(fontsize=8)

axes[-1].set_xlabel('t')
plt.suptitle('One-step-ahead forecasts: Fourier J=1, J=2, and factor form')
plt.tight_layout(); plt.show()

## 6. Monthly data — harmonic count selection

In [ ]:
# Simulate a monthly seasonal series (period=12) with 3 harmonics
spec_monthly = make_seasonal_factor(period=12, V=1.0, W_season=0.05)
sim_m = simulate(spec_monthly, n=240, seed=9)
y_m = sim_m.y

logliks = {}
dims = {}
for J in range(1, 7):
    spec_f = make_fourier_seasonal(period=12, n_harmonics=J, V=1.0, W_season=0.05)
    fr_f = kalman_filter(spec_f, y_m)
    logliks[J] = fr_f.loglik
    dims[J] = spec_f.d

Js = list(logliks.keys())
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.bar(Js, [logliks[j] for j in Js])
ax1.set(xlabel='J (harmonics)', ylabel='log marginal likelihood', title='Log-lik vs J')
ax2.bar(Js, [dims[j] for j in Js])
ax2.axhline(11, color='k', ls='--', label='factor form (d=11)')
ax2.set(xlabel='J', ylabel='state dimension', title='State dim vs J')
ax2.legend()
plt.tight_layout(); plt.show()

## 7. Exercises

**Exercise 1.** The full Fourier seasonal ($J = s/2$) should reproduce the factor-form seasonal **exactly** (up to reparametrisation) because both span the same spectral frequencies. Verify numerically: run both on the same data with matched $V$ and $W$ and compare log-likelihoods.

In [ ]:
# YOUR CODE HERE — use period=4 (simple case: J=2 = s/2)


**Exercise 2.** For the monthly series above, which $J$ maximises the log marginal likelihood? Explain why the optimal $J$ might be less than the maximum (J=6). What would happen if you tried $J=7$ for a period-12 series?

In [ ]:
# YOUR CODE HERE


**Exercise 3.** Combine a local-level component with a Fourier seasonal to build a level + seasonal model. Use `engine.models.combine()` and run on the monthly series. Compare its log-lik to the seasonal-only Fourier model.

In [ ]:
# YOUR CODE HERE
from engine.models import combine, make_local_level
